# 30. 数据合并与结构转换

<!-- module-learning-arc:start -->
> **Pandas 模块主线｜第 9 / 10 步：连接多张业务表**
>
> **持续应用背景：** 搭建电商履约异常追踪台：把订单、客户、商品和履约信息整理成安全合并的事实表，再生成趋势指标和异常工单。
>
> **承接上一阶段：** 分组、聚合与数据透视  →  **本章任务：** 数据合并与结构转换  →  **下一步：** 窗口计算与探索性分析
>
> **大作业连接：** 本章练习将成为《电商履约异常追踪台》的一部分，最终需要从多表质量审计走到订单粒度事实表、窗口趋势和可复核异常工单。
<!-- module-learning-arc:end -->


## 本章场景

做数据分析时，数据很少天生就是一张干净的"大宽表"。



## 本章目标

学完本章，你将能够：

- **理解**：理解 merge/join/concat 与长宽表转换。
- **操作**：能合并多表、做结构转换（melt/pivot）。
- **迁移**：能把多张订单表关联成一张完整事实表供分析。


## 30.1 核心概念

**背景引入**：做数据分析时，数据很少天生就是一张干净的"大宽表"。订单明细、客户档案、月度报表往往分散在不同的文件里，靠共同的键把它们接起来，或把行、列结构互相转一转换，才能凑成一份能直接分析的可用表。掌握合并与结构转换，就是把零散数据"拼"成可分析数据的基础能力。

- 连接前检查键唯一性和匹配率。
- concat负责轴向拼接，merge负责关系连接。
- 长表更适合分组统计和可视化。

> **直观类比**：merge 像把两本各自按“工号”记账的本子对到一起，靠共同键把同一人的记录并进一行；concat 则像把两本账“上/下、左/右摞起来”，只是物理拼接、不认键。


## 30.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| merge() | `pd.DataFrame()`、`orders.merge()` | 连接前确认键的唯一性和预期的关系类型。 | 连接键不唯一导致行数意外膨胀 |
| concat() | `pd.DataFrame()`、`pd.concat()` | concat沿行或列拼接，ignore_index可以重建连续索引。 | 连接后不检查未匹配记录 |
| melt() | `pd.DataFrame()`、`wide.melt()` | melt把多个指标列折叠为变量和值，适合后续分组和可视化。 | 重塑时遗漏标识列 |
| pivot() | `pd.DataFrame()`、`long.pivot()` | pivot要求每个行列组合只有一个值，否则需要使用pivot_table。 | 连接键不唯一导致行数意外膨胀 |
| stack() 与 unstack() | `pd.DataFrame()`、`table.stack()`、`stacked.unstack()` | stack把列层级压到索引，unstack把索引层级展开为列。 | 连接后不检查未匹配记录 |


## 30.3 示例 1：关系连接

validate参数可以验证预期的一对一或多对一关系。

**背景引入**：订单表里只有客户编号，要报“哪个城市的客户买了多少”，就得把订单表和客户表按 customer_id 接到一起。连接键不唯一是行数悄悄变多的头号元凶，拼之前先确认关系类型。

**讲解**：merge 用 on 指定连接键，how 决定保留哪一侧，validate 提前验证关系，indicator 标出每行来源。

- `how="left"` 保留左表所有订单，右边客户信息对得上就补上、对不上留空；
- `validate="many_to_one"` 在跑之前校验“右表键唯一”，不唯一直接报错，防止行数膨胀；
- `indicator=True` 生成 `_merge` 列，标注每行来自哪张表，帮助检查未匹配记录；
- **口诀**：on 找键、how 定侧、validate 防膨胀、indicator 查来源。


<!-- math-foundation:chapter-30 -->
### 数学推导｜连接完整率与粒度守恒

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜记录左表每一行是否匹配。** 令 $m_i=1$ 表示成功匹配，否则为 0。

$$
N_{matched}=\sum_{i=1}^{N_{left}}m_i
$$

**第 2 步｜得到匹配率。** $r_{match}=N_{matched}/N_{left}$。

**第 3 步｜检查一行会扩成几行。** 若左表第 $i$ 行在右表匹配到 $k_i$ 行，左连接后的行数为

$$
N_{after}=\sum_{i=1}^{N_{left}}\max(1,k_i)
$$

只有预期的一对一或多对一连接，才应有 $N_{after}=N_{left}$。

**把上面的关系收束为本章计算式：**

$$
r_{match}=\frac{N_{matched}}{N_{left}},\qquad N_{after}=N_{before}\ \text{（一对一或多对一左连接）}
$$

**符号解释：** $r_{match}$ 衡量左表记录成功匹配的比例。

**代码对应：** 使用 `indicator=True` 统计匹配状态，并用 `validate=` 声明连接关系。

**使用边界：** 行数守恒只适用于预期的一对一/多对一场景；明细表合并前常需先聚合。


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "order_id": ["A1", "A2", "A3"],
        "customer_id": ["U1", "U2", "U1"],
        "amount": [320, 880, 460],
    }
)
customers = pd.DataFrame(
    {
        "customer_id": ["U1", "U2"],
        "city": ["上海", "广州"],
    }
)
merged = orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)
print(merged)


## 30.4 示例 2：数据拼接

拼接后通常需要重新建立连续索引。

**背景引入**：每月的销售表单独存着，月底要把 1 月、2 月…从上到下摞成一张全年总表。直接拼出来的表索引会重复（两个表都有 0、1），取行、筛选时容易错位，重建连续索引是标配动作。

**讲解**：concat 默认沿行方向（axis=0）拼接多张表，ignore_index=True 重建连续索引。

- `pd.concat([january, february])` 默认上下拼接，要求两个表列名一致；
- `ignore_index=True` 让结果索引从 0 重新编号，避免重复索引带来的错位；
- 拼接后检查总行数是否等于各表行数之和，确认没有列对不上；
- **口诀**：上下拼用 concat，ignore_index 重排号，列名一致才拼得对。


In [ ]:
january = pd.DataFrame({"month": ["1月", "1月"], "sales": [120, 98]})
february = pd.DataFrame({"month": ["2月", "2月"], "sales": [150, 132]})
combined = pd.concat([january, february], ignore_index=True)
print(combined)


## 30.5 示例 3：宽表转长表

melt明确保留标识列，把多个指标列折叠为变量和值。

**背景引入**：表里“一月、二月、三月”各占一列，叫宽表——按月份分组、画趋势图都很别扭。把多个月份列折叠成“月份 + 销售额”两列的窄长表，后面分组、绘图都顺手；用 pivot 又能还原回宽表。

**讲解**：melt 用 id_vars 保留标识列，把其余指标列折叠成变量列（var_name）和值列（value_name）；pivot 反向还原宽表。

- `id_vars="region"` 是每行的“身份证”，一定别丢，否则还原时对不回去；
- `var_name="month"`、`value_name="sales"` 给折叠出来的两列起名，语义更清楚；
- `pivot` 要求每个“行 × 列”组合只有一个值，有重复就得改用 pivot_table；
- **口诀**：宽变长用 melt，id_vars 别忘；长变宽用 pivot，有重复找 pivot_table。


In [ ]:
wide = pd.DataFrame(
    {
        "region": ["华东", "华南"],
        "一月": [120, 98],
        "二月": [150, 132],
        "三月": [180, 145],
    }
)
long = wide.melt(id_vars="region", var_name="month", value_name="sales")
restored = long.pivot(index="region", columns="month", values="sales")
print(long)
print(restored)


## 30.6 核心操作独立示例

下面每个代码单元格只演示一个核心方法、函数或语法操作。请先阅读方法名称和任务说明，再单独运行当前单元格；示例尽量自带最小输入，不要求依赖前一个单元格留下的变量。


In [ ]:
# merge()
# 连接前确认键的唯一性和预期的关系类型。
import pandas as pd

orders = pd.DataFrame({"order_id": ["A1", "A2"], "customer_id": ["U1", "U2"]})
customers = pd.DataFrame({"customer_id": ["U1", "U2"], "city": ["上海", "广州"]})
print(
    orders.merge(
        customers, on="customer_id", how="left", validate="many_to_one"
    )
)


In [ ]:
# concat()
# concat沿行或列拼接，ignore_index可以重建连续索引。
import pandas as pd

first = pd.DataFrame({"month": ["1月"], "sales": [120]})
second = pd.DataFrame({"month": ["2月"], "sales": [150]})
print(pd.concat([first, second], ignore_index=True))


In [ ]:
# melt()
# melt把多个指标列折叠为变量和值，适合后续分组和可视化。
import pandas as pd

wide = pd.DataFrame(
    {"region": ["华东", "华南"], "一月": [120, 98], "二月": [150, 132]}
)
print(wide.melt(id_vars="region", var_name="month", value_name="sales"))


In [ ]:
# pivot()
# pivot要求每个行列组合只有一个值，否则需要使用pivot_table。
import pandas as pd

long = pd.DataFrame(
    {
        "region": ["华东", "华东", "华南", "华南"],
        "month": ["一月", "二月", "一月", "二月"],
        "sales": [120, 150, 98, 132],
    }
)
print(long.pivot(index="region", columns="month", values="sales"))


In [ ]:
# stack() 与 unstack()
# stack把列层级压到索引，unstack把索引层级展开为列。
import pandas as pd

table = pd.DataFrame({"一月": [120, 98], "二月": [150, 132]}, index=["华东", "华南"])
stacked = table.stack()
print(stacked)
print(stacked.unstack())


**练一练 26.6**：有两张销售表 `sales_a`、`sales_b`，一张学生表 `students`、一张成绩表 `scores`，还有一张宽表 `wide`。完成三件小任务：用 `concat` 沿行拼接两张销售表并重建索引；用 `merge` 把 `scores` 的分数按键接到 `students` 上；用 `melt` 把 `wide` 的一月、二月两列折叠成长表，并把结果打印出来。


In [ ]:
# 请在下方填写代码
import pandas as pd

sales_a = pd.DataFrame({"month": ["1月"], "sales": [120]})
sales_b = pd.DataFrame({"month": ["2月"], "sales": [150]})
# 第 1 步：用 concat 沿行拼接，并重建连续索引
combined = ___
print(combined)

students = pd.DataFrame({"stu_id": [1, 2], "name": ["小明", "小红"]})
scores = pd.DataFrame({"stu_id": [1, 2], "score": [88, 95]})
# 第 2 步：用 merge 把 score 接到 name 上
result = ___
print(result)

wide = pd.DataFrame(
    {"region": ["华东", "华南"], "一月": [120, 98], "二月": [150, 132]}
)
# 第 3 步：用 melt 把一月、二月两列折叠成 month / sales 两列
long = ___
print(long)


In [ ]:
import pandas as pd

sales_a = pd.DataFrame({"month": ["1月"], "sales": [120]})
sales_b = pd.DataFrame({"month": ["2月"], "sales": [150]})
# 第 1 步：concat 沿行拼接并重建索引
combined = pd.concat([sales_a, sales_b], ignore_index=True)
print(combined)

students = pd.DataFrame({"stu_id": [1, 2], "name": ["小明", "小红"]})
scores = pd.DataFrame({"stu_id": [1, 2], "score": [88, 95]})
# 第 2 步：merge 按键连接
result = students.merge(scores, on="stu_id")
print(result)

wide = pd.DataFrame(
    {"region": ["华东", "华南"], "一月": [120, 98], "二月": [150, 132]}
)
# 第 3 步：melt 宽表转长表
long = wide.melt(id_vars="region", var_name="month", value_name="sales")
print(long)


## 30.7 公开大型数据实战

下面使用 UCI Machine Learning Repository 的 Online Retail 公开数据集。原始数据包含 541,909 条英国在线零售交易，本课程使用固定随机种子抽取的 200,000 行子集。分析时在完整子集上计算，只展示摘要或少量样本。


In [ ]:
import numpy as np
import pandas as pd

# UCI Machine Learning Repository: Online Retail
# 原始数据 541,909 行；课程使用固定随机种子抽取的 200,000 行子集。
data_url = "/datasets/uci_online_retail_200k.csv"
large_orders = pd.read_csv(
    data_url,
    parse_dates=["InvoiceDate"],
    dtype={
        "InvoiceNo": "string",
        "StockCode": "string",
        "Description": "string",
        "Country": "category",
    },
).rename(
    columns={
        "InvoiceNo": "order_id",
        "StockCode": "stock_code",
        "Description": "description",
        "Quantity": "quantity",
        "InvoiceDate": "order_time",
        "UnitPrice": "unit_price",
        "CustomerID": "customer_id",
        "Country": "country",
    }
)
large_orders["sales"] = (
    large_orders["quantity"] * large_orders["unit_price"]
).round(2)
large_orders["status"] = np.where(
    large_orders["order_id"].str.startswith("C")
    | (large_orders["quantity"] < 0),
    "取消/退货",
    "完成",
)
print("UCI Online Retail 公开数据：")
print(f"  {len(large_orders):,} 行 × {large_orders.shape[1]} 列")
print(
    "内存占用：", f"{large_orders.memory_usage(deep=True).sum() / 1024**2:.1f} MB"
)
large_orders.head()


In [ ]:
customer_dimension = (
    large_orders.dropna(subset=["customer_id"])
    .groupby("customer_id", as_index=False)
    .agg(home_country=("country", "first"), first_order=("order_time", "min"))
)
enriched = large_orders.merge(
    customer_dimension,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)
country_sales = (
    enriched.groupby("home_country", observed=True)
    .agg(
        订单行数=("order_id", "size"),
        销售额=("sales", "sum"),
        客户数=("customer_id", "nunique"),
    )
    .sort_values("销售额", ascending=False)
)
print("连接结果：", enriched.shape, "缺失客户维度：", (enriched["_merge"] != "both").sum())
display(country_sales.head(10).round(2))


## 30.8 独立迁移练习

替换一个字段或分组口径，并核对处理前后的行数与粒度。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 30.9 本章实训：分组汇总与粒度

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "region": ["华东", "华东", "华南", "华南"],
        "channel": ["线上", "线下", "线上", "线下"],
        "sales": [120, 80, 150, 100],
    }
)
summary = orders.groupby("region", as_index=False)["sales"].sum()
print(summary)
print("汇总表每一行代表一个地区")


### 30.9.1 第一个结果怎么读

先确认明细表一行代表一笔订单，再确认汇总表一行代表一个地区。`groupby` 的字段决定结果的粒度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。



In [ ]:
orders["sales_level"] = orders["sales"].map(
    lambda value: "高" if value >= 120 else "普通"
)
print(orders)
print(orders["sales_level"].value_counts())


### 30.9.2 第二个结果怎么读

第二个实验只增加一个分类列，不改变原始销售额。练习解释：什么时候应该新增列，什么时候应该直接筛选行？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。



## 30.10 错误恢复：脏数据转换怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.Series(["12", "unknown", "18", ""])
converted = pd.to_numeric(raw, errors="coerce")
print("转换结果：")
print(converted)
print("无法转换的数量：", converted.isna().sum())
print("后续可以选择删除、填充或回查原始值。")


### 30.10.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

errors="coerce" 会把无法转换的值记录为缺失，适合先完成质量盘点；不要在没有统计数量前直接删除。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。



## 30.11 易错点提醒

- 连接键不唯一导致行数意外膨胀
- 连接后不检查未匹配记录
- 重塑时遗漏标识列


## 30.12 练习与作业

1. 连接订单表与商品表
2. 计算订单行金额
3. 将月度宽表转换为长表

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 30.13 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“连接订单表与商品表”。
2. **独立完成**：不复制示例代码，完成“计算订单行金额”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“将月度宽表转换为长表”，用一两句话说明你修改了什么。

### 30.13.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 30.13.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import pandas as pd

# TODO: 连接订单表与商品表
# TODO: 计算订单行金额
# TODO：请在下方完成 —— 26.13 练习与作业 1. 连接订单表与商品表 2. 计算订单行金额 3. 将月度宽表转换为长表 提交前检查：代码可从


In [ ]:
import pandas as pd

items = pd.DataFrame({"product_id": ["P1", "P2"], "price": [299, 129]})
order_lines = pd.DataFrame(
    {
        "order_id": ["A1", "A1", "A2"],
        "product_id": ["P1", "P2", "P2"],
        "quantity": [1, 2, 3],
    }
)
result = order_lines.merge(items, on="product_id", validate="many_to_one")
result["line_amount"] = result["price"] * result["quantity"]
print(result)
print(result.groupby("order_id")["line_amount"].sum())


## 30.14 小结

通过merge、concat、melt、stack和unstack整合并重塑表格。

**迁移思考**：

1. 如果订单表和商品表连接后行数突然增加了10倍，最可能的原因是什么？如何诊断？
2. 为什么长表更适合分组统计和可视化？宽表适合什么场景？



### 30.14.1 你已经掌握

- 按键连接表格
- 纵向与横向拼接
- 宽表转长表
- 在索引层级间重塑



### 30.14.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。



### 30.14.3 需要注意

- 连接键不唯一导致行数意外膨胀
- 连接后不检查未匹配记录
- 重塑时遗漏标识列



### 30.14.4 完成检查

- [ ] 能够按键连接表格
- [ ] 能够纵向与横向拼接
- [ ] 能够宽表转长表
- [ ] 能够在索引层级间重塑



### 30.14.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。

